# Motor Insurance Analytics
## Business Understanding
A motor insurance company needs to understand:
- Policy activity (Active / Expired / Cancelled / Renewed)
- Premium collections by policy type
- Claim volume, approval / rejection / pending status
- Claim severity by region, vehicle type, and customer segment
- Settlement performance
- Risk indicators: claim-to-premium ratio, paid-claim ratio

Objective: convert raw operational data into a validated, cleaned, KPI-driven,
insight-rich dataset used by a Streamlit dashboard.

In [2]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sys.path.append(os.path.abspath(".."))  # so we can import src

from src.data_loader import load_all_data
from src.validation import validate_all_tables
from src.data_cleaner import clean_data
from src.insurance_analysis import *
from src.visualization import *
from src.insights import generate_business_recommendations

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)

In [3]:
data = load_all_data("../data/raw")
for k, v in data.items():
    print(f"{k:10s} shape={v.shape}")

2026-10-09 11:54:35 | INFO     | motor_insurance | [PIPELINE] Loading all tables from ../data/raw
2026-10-09 11:54:36 | INFO     | motor_insurance | [PIPELINE] Loaded customers.csv -> shape (20000, 8)
2026-10-09 11:54:36 | INFO     | motor_insurance | [PIPELINE] Loaded vehicles.csv -> shape (25000, 8)
2026-10-09 11:54:36 | INFO     | motor_insurance | [PIPELINE] Loaded policies.csv -> shape (25000, 10)
2026-10-09 11:54:36 | INFO     | motor_insurance | [PIPELINE] Loaded claims.csv -> shape (10000, 12)
2026-10-09 11:54:36 | INFO     | motor_insurance | [PIPELINE] Loaded payments.csv -> shape (7000, 7)


customers  shape=(20000, 8)
vehicles   shape=(25000, 8)
policies   shape=(25000, 10)
claims     shape=(10000, 12)
payments   shape=(7000, 7)


In [4]:
print("customers keys:", data["customers"].columns.tolist())
print("vehicles  keys:", data["vehicles"].columns.tolist())
print("policies  keys:", data["policies"].columns.tolist())
print("claims    keys:", data["claims"].columns.tolist())
print("payments  keys:", data["payments"].columns.tolist())
# Relationship: policies.customer_id -> customers.customer_id, etc.

customers keys: ['customer_id', 'customer_name', 'gender', 'date_of_birth', 'age', 'city', 'state', 'occupation']
vehicles  keys: ['vehicle_id', 'customer_id', 'vehicle_type', 'vehicle_make', 'vehicle_model', 'vehicle_year', 'fuel_type', 'vehicle_value']
policies  keys: ['policy_id', 'customer_id', 'vehicle_id', 'policy_start_date', 'policy_end_date', 'policy_type', 'premium_amount', 'coverage_amount', 'policy_status', 'payment_frequency']
claims    keys: ['claim_id', 'policy_id', 'customer_id', 'claim_date', 'accident_date', 'claim_type', 'accident_location', 'claim_amount', 'claim_status', 'approval_date', 'settlement_date', 'damage_severity']
payments  keys: ['payment_id', 'claim_id', 'policy_id', 'payment_date', 'payment_amount', 'payment_status', 'payment_method']


In [5]:
for k, v in data.items():
    print(f"\n===== {k.upper()} =====")
    print(v.info())
    print(v.describe(include="all").T.head(10))


===== CUSTOMERS =====
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    20000 non-null  str  
 1   customer_name  20000 non-null  str  
 2   gender         20000 non-null  str  
 3   date_of_birth  20000 non-null  str  
 4   age            20000 non-null  int64
 5   city           20000 non-null  str  
 6   state          20000 non-null  str  
 7   occupation     20000 non-null  str  
dtypes: int64(1), str(7)
memory usage: 2.5 MB
None
                 count unique                top   freq      mean        std  \
customer_id      20000  20000        CUST_000001      1       NaN        NaN   
customer_name    20000  19276        Charvi Dara      4       NaN        NaN   
gender           20000      2               Male  12451       NaN        NaN   
date_of_birth    20000  12460         1985-07-08      7       NaN        NaN   
age      

In [6]:
for k, v in data.items():
    missing = v.isna().sum()
    dup = v.duplicated().sum()
    print(f"{k:10s} | missing total={missing.sum():5d} | duplicated={dup}")

customers  | missing total=    0 | duplicated=0
vehicles   | missing total=    0 | duplicated=0
policies   | missing total=    0 | duplicated=0
claims     | missing total= 9692 | duplicated=0
payments   | missing total=    0 | duplicated=0


In [7]:
ok = validate_all_tables(data)
print("Validation passed:", ok)
data = clean_data(data)

2026-10-09 11:56:21 | INFO     | motor_insurance | [PIPELINE] Starting validation of all tables
2026-10-09 11:56:22 | INFO     | motor_insurance | [PIPELINE] All validations passed
2026-10-09 11:56:22 | INFO     | motor_insurance | [PIPELINE] Running full cleaning pipeline
2026-10-09 11:56:22 | INFO     | motor_insurance | [PIPELINE] Missing values handled
2026-10-09 11:56:22 | INFO     | motor_insurance | [PIPELINE] customers: removed 0 duplicate rows


Validation passed: True


2026-10-09 11:56:22 | INFO     | motor_insurance | [PIPELINE] vehicles: removed 0 duplicate rows
2026-10-09 11:56:23 | INFO     | motor_insurance | [PIPELINE] policies: removed 0 duplicate rows
2026-10-09 11:56:23 | INFO     | motor_insurance | [PIPELINE] claims: removed 0 duplicate rows
2026-10-09 11:56:23 | INFO     | motor_insurance | [PIPELINE] payments: removed 0 duplicate rows
2026-10-09 11:56:23 | INFO     | motor_insurance | [PIPELINE] Text columns cleaned
2026-10-09 11:56:24 | INFO     | motor_insurance | [PIPELINE] Date columns converted
2026-10-09 11:56:24 | INFO     | motor_insurance | [PIPELINE] Numeric columns validated
2026-10-09 11:56:24 | INFO     | motor_insurance | [PIPELINE] Business rules enforced
2026-10-09 11:56:24 | INFO     | motor_insurance | [PIPELINE] Derived columns created


In [8]:
data["policies"][["policy_id", "policy_duration_days", "policy_active_flag"]].head()
data["claims"][["claim_id", "claim_processing_days",
                "claim_settlement_days", "claim_to_premium_ratio"]].head()

,claim_id,claim_processing_days,claim_settlement_days,claim_to_premium_ratio
0,Clm_000001,11.0,18.0,2.915769
1,Clm_000002,NaN,NaN,12.405855
2,Clm_000003,NaN,NaN,3.877266
3,Clm_000004,4.0,NaN,55.910971
4,Clm_000005,11.0,NaN,58.708785


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
sns.histplot(data["policies"]["premium_amount"], kde=True, ax=axes[0,0], color="teal")
axes[0,0].set_title("Premium Distribution")
sns.histplot(data["claims"]["claim_amount"], kde=True, ax=axes[0,1], color="crimson")
axes[0,1].set_title("Claim Amount Distribution")
sns.histplot(data["vehicles"]["vehicle_age"], ax=axes[0,2], color="orange")
axes[0,2].set_title("Vehicle Age Distribution")
sns.histplot(data["customers"]["customer_age"], ax=axes[1,0], color="purple")
axes[1,0].set_title("Customer Age Distribution")
data["policies"]["policy_status"].value_counts().plot.bar(ax=axes[1,1], color="steelblue")
axes[1,1].set_title("Policy Status")
data["claims"]["claim_status"].value_counts().plot.bar(ax=axes[1,2], color="green")
axes[1,2].set_title("Claim Status")
plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
policy_groups = data["policies"].groupby("policy_type", sort=True)["premium_amount"]
policy_labels = list(policy_groups.groups)
axes[0, 0].boxplot([policy_groups.get_group(label).dropna().to_numpy() for label in policy_labels])
axes[0, 0].set_xticks(range(1, len(policy_labels) + 1), labels=policy_labels)
axes[0,0].set_title("Premium vs Policy Type")
claim_policies = data["policies"][["policy_id", "vehicle_id"]]
df = data["claims"].merge(claim_policies, on="policy_id", how="left")
df = df.merge(data["vehicles"][["vehicle_id", "vehicle_type", "vehicle_age"]], on="vehicle_id", how="left")
vehicle_groups = df.groupby("vehicle_type", sort=True)["claim_amount"]
vehicle_labels = list(vehicle_groups.groups)
axes[0, 1].boxplot([vehicle_groups.get_group(label).dropna().to_numpy() for label in vehicle_labels])
axes[0, 1].set_xticks(range(1, len(vehicle_labels) + 1), labels=vehicle_labels)
axes[0,1].set_title("Claim Amount vs Vehicle Type")
sns.scatterplot(x="vehicle_age", y="claim_amount", data=df, alpha=0.4, ax=axes[1,0])
axes[1,0].set_title("Vehicle Age vs Claim Amount")
df2 = data["claims"].merge(data["customers"][["customer_id","customer_age"]],
                           on="customer_id", how="left")
sns.scatterplot(x="customer_age", y="claim_amount", data=df2, alpha=0.4, ax=axes[1,1])
axes[1,1].set_title("Customer Age vs Claim Amount")
plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
claim_policies = data["policies"][["policy_id", "vehicle_id"]]
claims_with_vehicles = data["claims"].merge(claim_policies, on="policy_id", how="left")
claims_with_vehicles = claims_with_vehicles.merge(
    data["vehicles"][["vehicle_id", "vehicle_type"]], on="vehicle_id", how="left")
pivot = (claims_with_vehicles
         .pivot_table(index="vehicle_type", columns="damage_severity",
                      values="claim_amount", aggfunc="mean"))
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax)
ax.set_title("Avg Claim Amount: Vehicle Type × Damage Severity")
fig.tight_layout()
display(fig)
plt.close(fig)

In [12]:
print("Premium mean/median/std:",
      data["policies"]["premium_amount"].mean(),
      data["policies"]["premium_amount"].median(),
      data["policies"]["premium_amount"].std())

print("Claim mean/median/std:",
      data["claims"]["claim_amount"].mean(),
      data["claims"]["claim_amount"].median(),
      data["claims"]["claim_amount"].std())

print("Settlement days mean/std:",
      data["claims"]["claim_settlement_days"].mean(),
      data["claims"]["claim_settlement_days"].std())

corr = data["claims"][["claim_amount","claim_processing_days",
                       "claim_settlement_days","claim_to_premium_ratio"]].corr()
print("\nCorrelation matrix:\n", corr)

Premium mean/median/std: 31825.99951806263 28918.7 18096.126414551534
Claim mean/median/std: 288543.64406500006 205040.095 263048.1020793543
Settlement days mean/std: 23.925744093280148 9.846863527188422

Correlation matrix:
                         claim_amount  claim_processing_days  \
claim_amount                1.000000              -0.006950   
claim_processing_days      -0.006950               1.000000   
claim_settlement_days       0.019503               0.494998   
claim_to_premium_ratio      0.645253              -0.016573   

                        claim_settlement_days  claim_to_premium_ratio  
claim_amount                         0.019503                0.645253  
claim_processing_days                0.494998               -0.016573  
claim_settlement_days                1.000000                0.000098  
claim_to_premium_ratio               0.000098                1.000000  


In [13]:
kpis = {
    "Total Customers":     calculate_total_customers(data),
    "Total Policies":      calculate_total_policies(data),
    "Active Policies":     calculate_active_policies(data),
    "Expired Policies":    calculate_expired_policies(data),
    "Cancelled Policies":  calculate_cancelled_policies(data),
    "Renewed Policies":    calculate_renewed_policies(data),
    "Total Premium":       calculate_total_premium(data),
    "Avg Premium":         calculate_average_premium(data),
    "Total Claims":        calculate_total_claims(data),
    "Approved Claims":     calculate_approved_claims(data),
    "Rejected Claims":     calculate_rejected_claims(data),
    "Pending Claims":      calculate_pending_claims(data),
    "Total Claim Amount":  calculate_total_claim_amount(data),
    "Avg Claim Amount":    calculate_average_claim_amount(data),
    "Approval Rate %":     round(calculate_claim_approval_rate(data), 2),
    "Rejection Rate %":    round(calculate_claim_rejection_rate(data), 2),
    "Avg Settlement Days": round(calculate_average_claim_settlement_days(data), 2),
    "Claim/Premium Ratio": round(calculate_claim_to_premium_ratio(data), 3),
    "Paid Claim Ratio":    round(calculate_paid_claim_ratio(data), 3),
}
pd.DataFrame(kpis.items(), columns=["KPI", "Value"])

,KPI,Value
0,Total Customers,2.000000e+04
1,Total Policies,2.494100e+04
2,Active Policies,7.551000e+03
3,Expired Policies,1.046400e+04
4,Cancelled Policies,2.610000e+02
5,Renewed Policies,6.665000e+03
6,Total Premium,7.937723e+08
7,Avg Premium,3.182600e+04
8,Total Claims,1.000000e+04
9,Approved Claims,3.790000e+03


In [14]:
print("Q1: Total policies:", calculate_total_policies(data))
print("Q2: Dominant policy type:\n", data["policies"]["policy_type"].value_counts().head())
print("Q3: Premium by policy type:\n", calculate_premium_by_policy_type(data))
print("Q4: Claim status breakdown:\n", analyze_claim_status(data))
print("Q5: Avg claim by claim type:\n", analyze_claim_type(data))
print("Q6: Claims by region:\n", data["claims"]["accident_location"].value_counts().head())
print("Q7: Vehicle type risk:\n", analyze_vehicle_claims(data))

Q1: Total policies: 24941
Q2: Dominant policy type:
 policy_type
Comprehensive        15055
Third Party           6198
Zero Depreciation     3688
Name: count, dtype: int64
Q3: Premium by policy type:
 policy_type
Comprehensive        5.439424e+08
Zero Depreciation    1.744929e+08
Third Party          7.533696e+07
Name: premium_amount, dtype: float64
Q4: Claim status breakdown:
       count  count
0  Approved   3790
1   Settled   3259
2   Pending   1791
3  Rejected   1160
Q5: Avg claim by claim type:
            claim_type  count  total_amount     avg_amount
0            Accident   4838  1.377621e+09  284750.185763
4  Third Party Damage   2022  5.954065e+08  294464.138348
3               Theft   1585  4.551719e+08  287174.721464
2    Natural Disaster    796  2.300795e+08  289044.594246
1                Fire    759  2.271571e+08  299284.747892
Q6: Claims by region:
 accident_location
National Highway    1703
Parking Area        1687
City Road           1673
Highway             1673
Resid

In [ ]:
plot_policy_status(data)
plot_policy_type_distribution(data)
plot_premium_by_policy_type(data)
plot_monthly_claims(data)
plot_claim_status(data)
plot_claim_type_distribution(data)
plot_claim_amount_distribution(data)
plot_claim_amount_by_vehicle_type(data)
plot_claims_by_region(data)
plot_average_settlement_time(data)
plot_claim_to_premium_by_policy_type(data)
plot_vehicle_age_vs_claim_amount(data)
plot_customer_age_vs_claim_amount(data)
plot_damage_severity_distribution(data)
plot_payment_status(data)
plot_monthly_premium_trend(data)

In [16]:
cards = generate_business_recommendations(data)
for i, c in enumerate(cards, 1):
    print(f"\n--- Insight {i} ---")
    for k, v in c.items():
        print(f"{k}: {v}")


--- Insight 1 ---
PATTERN: Average claim amount is higher than the associated premium.
EVIDENCE: Average claim-to-premium ratio: 12.84 (1284%).
BUSINESS_MEANING: Claims are costly relative to the premiums associated with them.
RECOMMENDATION: Review pricing and underwriting for the affected portfolio segments.

--- Insight 2 ---
PATTERN: Pending claims are awaiting a decision.
EVIDENCE: 1,791 of 10,000 claims are pending (17.9%).
BUSINESS_MEANING: Unresolved claims may increase customer wait times and service workload.
RECOMMENDATION: Review pending claims regularly and prioritize older submissions.

--- Insight 3 ---
PATTERN: A notable share of the selected policies has expired.
EVIDENCE: 10,464 of 24,941 policies are expired (42.0%).
BUSINESS_MEANING: Expired policies may represent missed renewal opportunities.
RECOMMENDATION: Start renewal outreach before policy end dates and follow up on recent expirations.


In [17]:
print("INSIGHT 1: Premium concentration")
print(calculate_premium_by_policy_type(data).head(3))

print("\nINSIGHT 2: Claims concentration by type")
print(analyze_claim_type(data).head(3))

print("\nINSIGHT 3: Highest risk policy type")
print(data["claims"].groupby("policy_type")["claim_to_premium_ratio"]
      .mean().sort_values(ascending=False))

INSIGHT 1: Premium concentration
policy_type
Comprehensive        5.439424e+08
Zero Depreciation    1.744929e+08
Third Party          7.533696e+07
Name: premium_amount, dtype: float64

INSIGHT 2: Claims concentration by type
           claim_type  count  total_amount     avg_amount
0            Accident   4838  1.377621e+09  284750.185763
4  Third Party Damage   2022  5.954065e+08  294464.138348
3               Theft   1585  4.551719e+08  287174.721464

INSIGHT 3: Highest risk policy type
policy_type
Third Party          30.015873
Comprehensive         7.419117
Zero Depreciation     6.133370
Name: claim_to_premium_ratio, dtype: float64


In [18]:
recommendations = [
    "Re-price policy types with claim-to-premium ratio > 1.",
    "Automate settlement for claim types with avg settlement > 30 days.",
    "Target renewal campaigns to Expired policy holders.",
    "Focus retention on the top-premium customer age segment.",
    "Deep-dive into the vehicle type contributing the highest claim value.",
]
for r in recommendations:
    print("-", r)

- Re-price policy types with claim-to-premium ratio > 1.
- Automate settlement for claim types with avg settlement > 30 days.
- Target renewal campaigns to Expired policy holders.
- Focus retention on the top-premium customer age segment.
- Deep-dive into the vehicle type contributing the highest claim value.


## Final Conclusion
------------------------
The raw motor-insurance data has been validated, cleaned, and enriched with derived
columns. KPIs and visualizations reveal clear patterns in premium concentration,
claim severity, settlement time, and risk ratios.

The outputs have been exposed through a Streamlit dashboard and prepared for
Docker-based deployment to AWS EC2 via Amazon ECR.

This notebook is the analytical backbone of the project; the production pipeline is
`run_pipeline.py` and the interactive layer is `app/streamlit_app.py`.